# AWS S3 Data Intelligence with Cortex AI Functions, Search, Analyst & Agents

## Lab Overview

In this lab you will build a complete data intelligence pipeline:

1. **Auto-ingest** files from Amazon S3 into Snowflake using Snowpipe
2. **Process** them through 11 Cortex AI functions (parse, transcribe, extract, classify, sentiment, summarize, translate, redact, complete, embed)
3. **Search** across all content using Cortex Search (semantic retrieval)
4. **Query** structured data using Cortex Analyst (natural-language-to-SQL)
5. **Chat** with a unified Cortex Agent via Snowflake Intelligence

**Duration:** ~90 minutes

**Before you start this notebook**, complete the AWS setup:
- Create an S3 bucket with prefixes `healthcare/pdfs/`, `healthcare/txt/`, `healthcare/audio/`
- Create an IAM role for Snowflake access
- Upload the sample files from the `sample_files/` folder to your S3 bucket

See the full AWS instructions in `LAB_GUIDE.md` (Steps 2, 3, 5, 7, 8).

---
## Step 1: Create Database, Schemas & Warehouse

We create a dedicated database with three schemas:
- **RAW** — Landing zone for file metadata from S3
- **PROCESSED** — AI-enriched intelligence tables
- **ANALYTICS** — Structured data, semantic view, and agent

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE DATABASE HEALTHCARE_AI_DEMO
  COMMENT = 'Healthcare AI Intelligence Pipeline';

CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.RAW;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.PROCESSED;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.ANALYTICS;

CREATE OR REPLACE WAREHOUSE HEALTHCARE_AI_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE HEALTHCARE_AI_WH;

---
## Step 2: Create Storage Integration

A **storage integration** establishes trust between Snowflake and your AWS account. This allows Snowflake to read files from your S3 bucket without embedding credentials in SQL.

**Action required:** Replace `<YOUR_BUCKET_NAME>` and `<YOUR_AWS_IAM_ROLE_ARN>` with your actual values.

In [ ]:
CREATE OR REPLACE STORAGE INTEGRATION HEALTHCARE_S3_INTEGRATION
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'S3'
  ENABLED = TRUE
  STORAGE_AWS_ROLE_ARN = '<YOUR_AWS_IAM_ROLE_ARN>'
  STORAGE_ALLOWED_LOCATIONS = ('s3://<YOUR_BUCKET_NAME>/healthcare/');

### Retrieve Snowflake's IAM User ARN & External ID

The output below contains two values you need to copy:
- `STORAGE_AWS_IAM_USER_ARN` — Snowflake's IAM user that will assume your role
- `STORAGE_AWS_EXTERNAL_ID` — A unique identifier for this integration

**After copying these values**, go to AWS Console → IAM → Roles → your role → Trust relationships → Edit, and paste them into the trust policy.

In [ ]:
DESCRIBE INTEGRATION HEALTHCARE_S3_INTEGRATION;

---
## Step 3: Create External Stages

External stages point to specific S3 prefixes. They allow Snowflake to reference files in S3 without copying them — the files stay in S3 and are read on-demand by AI functions.

**Action required:** Replace `<YOUR_BUCKET_NAME>` below.

**Important:** Wait at least 30 seconds after updating the IAM trust policy before running this cell.

In [ ]:
CREATE OR REPLACE STAGE HEALTHCARE_AI_DEMO.RAW.S3_MEDICAL_DOCS
  URL = 's3://<YOUR_BUCKET_NAME>/healthcare/pdfs/'
  STORAGE_INTEGRATION = HEALTHCARE_S3_INTEGRATION
  DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE);

CREATE OR REPLACE STAGE HEALTHCARE_AI_DEMO.RAW.S3_MEDICAL_TXT
  URL = 's3://<YOUR_BUCKET_NAME>/healthcare/txt/'
  STORAGE_INTEGRATION = HEALTHCARE_S3_INTEGRATION
  DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE);

CREATE OR REPLACE STAGE HEALTHCARE_AI_DEMO.RAW.S3_MEDICAL_AUDIO
  URL = 's3://<YOUR_BUCKET_NAME>/healthcare/audio/'
  STORAGE_INTEGRATION = HEALTHCARE_S3_INTEGRATION
  DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE);

CREATE OR REPLACE STAGE HEALTHCARE_AI_DEMO.ANALYTICS.SEMANTIC_MODELS
  DIRECTORY = (ENABLE = TRUE);

### Verify S3 Access

If this returns your uploaded files (or an empty result without errors), the integration is working correctly. If you get an "Access Denied" error, double-check the IAM trust policy values.

In [ ]:
LIST @HEALTHCARE_AI_DEMO.RAW.S3_MEDICAL_DOCS;

---
## Step 4: Create Snowpipes & Stream

**Snowpipe** provides automated file ingestion from S3. When configured with `AUTO_INGEST = TRUE`, Snowflake creates a managed SQS queue. S3 event notifications send messages to this queue whenever new files are uploaded, triggering Snowpipe to load metadata.

We only ingest **file metadata** (name, path, type, timestamp) — not file content. The actual files stay in S3 and are read directly by AI functions via `TO_FILE()`.

A **stream** captures changes to the FILES_LOG table, enabling event-driven processing of new arrivals.

In [ ]:
USE DATABASE HEALTHCARE_AI_DEMO;
USE SCHEMA RAW;

CREATE OR REPLACE TABLE RAW.FILES_LOG (
    FILE_ID       NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_NAME     VARCHAR NOT NULL,
    FILE_PATH     VARCHAR NOT NULL,
    FILE_TYPE     VARCHAR(10) NOT NULL,
    S3_EVENT_TIME TIMESTAMP_NTZ,
    LANDED_AT     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    IS_PROCESSED  BOOLEAN DEFAULT FALSE,
    PROCESSED_AT  TIMESTAMP_NTZ
);

CREATE OR REPLACE FILE FORMAT RAW.METADATA_ONLY_FORMAT
  TYPE = 'CSV' RECORD_DELIMITER = NONE FIELD_DELIMITER = NONE;

CREATE OR REPLACE PIPE RAW.PIPE_MEDICAL_DOCS AUTO_INGEST = TRUE AS
  COPY INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  FROM (
    SELECT METADATA$FILENAME, METADATA$FILENAME, 'PDF', METADATA$START_SCAN_TIME
    FROM @RAW.S3_MEDICAL_DOCS
  )
  FILE_FORMAT = (FORMAT_NAME = 'RAW.METADATA_ONLY_FORMAT');

CREATE OR REPLACE PIPE RAW.PIPE_MEDICAL_TXT AUTO_INGEST = TRUE AS
  COPY INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  FROM (
    SELECT METADATA$FILENAME, METADATA$FILENAME, 'TXT', METADATA$START_SCAN_TIME
    FROM @RAW.S3_MEDICAL_TXT
  )
  FILE_FORMAT = (FORMAT_NAME = 'RAW.METADATA_ONLY_FORMAT');

CREATE OR REPLACE PIPE RAW.PIPE_MEDICAL_AUDIO AUTO_INGEST = TRUE AS
  COPY INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  FROM (
    SELECT METADATA$FILENAME, METADATA$FILENAME,
      CASE WHEN METADATA$FILENAME ILIKE '%.mp3' THEN 'MP3' ELSE 'WAV' END,
      METADATA$START_SCAN_TIME
    FROM @RAW.S3_MEDICAL_AUDIO
  )
  FILE_FORMAT = (FORMAT_NAME = 'RAW.METADATA_ONLY_FORMAT');

CREATE OR REPLACE STREAM RAW.FILES_LOG_STREAM
  ON TABLE RAW.FILES_LOG APPEND_ONLY = TRUE;

### Get SQS Queue ARN

The `notification_channel` column contains the Snowflake-managed SQS queue ARN. You need this to configure S3 event notifications.

**Copy this ARN** — all three pipes share the same queue.

In [ ]:
SHOW PIPES IN SCHEMA RAW;

---
## PAUSE: Configure S3 Event Notifications

Go to **AWS Console → S3 → Your bucket → Properties → Event notifications** and create 3 event notifications:

| Name | Prefix filter | Events | Destination |
|------|--------------|--------|-------------|
| `snowpipe-pdfs` | `healthcare/pdfs/` | All object create events | SQS queue → paste the ARN from above |
| `snowpipe-txt` | `healthcare/txt/` | All object create events | SQS queue → paste the ARN from above |
| `snowpipe-audio` | `healthcare/audio/` | All object create events | SQS queue → paste the ARN from above |

**Continue below once the event notifications are saved.**

---
## Step 5: Trigger File Ingestion

`ALTER PIPE ... REFRESH` tells Snowpipe to scan the stage and load any files already present. For future uploads, the SQS event notifications will trigger ingestion automatically.

In [ ]:
ALTER PIPE RAW.PIPE_MEDICAL_DOCS REFRESH;
ALTER PIPE RAW.PIPE_MEDICAL_TXT REFRESH;
ALTER PIPE RAW.PIPE_MEDICAL_AUDIO REFRESH;

### Verify Files Landed

Wait 1-2 minutes for Snowpipe to process, then run the check below.

**Expected:** 6 PDF, 4 TXT, 5 WAV/MP3 (15 total).

In [ ]:
SELECT FILE_TYPE, COUNT(*) AS FILES FROM RAW.FILES_LOG GROUP BY FILE_TYPE;

### Fix FILE_NAME Prefix

Snowpipe stores the full S3 path relative to the bucket root (e.g., `healthcare/pdfs/report.pdf`). The AI functions need just the filename relative to the stage (e.g., `report.pdf`). This update strips the prefix.

In [ ]:
UPDATE RAW.FILES_LOG
SET FILE_NAME = REGEXP_REPLACE(FILE_NAME, '^healthcare/(pdfs|txt|audio)/', '')
WHERE FILE_NAME LIKE 'healthcare/%';

---
## Step 6: Create AI Processing Tables

Each file type gets its own intelligence table storing the output of all AI functions. These tables will hold: parsed text, extracted fields, document categories, sentiment scores, summaries, translations, PII-redacted text, LLM-generated insights, and vector embeddings.

In [ ]:
USE SCHEMA PROCESSED;

CREATE OR REPLACE TABLE PROCESSED.PDF_INTELLIGENCE (
    DOC_ID NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_ID NUMBER NOT NULL, FILE_NAME VARCHAR NOT NULL,
    PROCESSED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    PARSED_TEXT VARCHAR(16777216), PARSED_LAYOUT VARIANT,
    EXTRACTED_FIELDS VARIANT, DOC_CATEGORY VARCHAR(100),
    DOC_CATEGORY_CONFIDENCE FLOAT, SENTIMENT_SCORE FLOAT,
    SENTIMENT_DIMENSIONS VARIANT, SUMMARY VARCHAR(10000),
    DETECTED_LANGUAGE VARCHAR(50), TRANSLATED_TEXT VARCHAR(16777216),
    REDACTED_TEXT VARCHAR(16777216), KEY_INSIGHTS VARCHAR(10000),
    EMBEDDING VECTOR(FLOAT, 768)
);

CREATE OR REPLACE TABLE PROCESSED.TXT_INTELLIGENCE (
    TXT_ID NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_ID NUMBER NOT NULL, FILE_NAME VARCHAR NOT NULL,
    PROCESSED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    RAW_TEXT VARCHAR(16777216), EXTRACTED_FIELDS VARIANT,
    DOC_CATEGORY VARCHAR(100), DOC_CATEGORY_CONFIDENCE FLOAT,
    SENTIMENT_SCORE FLOAT, SENTIMENT_DIMENSIONS VARIANT,
    SUMMARY VARCHAR(10000), DETECTED_LANGUAGE VARCHAR(50),
    TRANSLATED_TEXT VARCHAR(16777216), REDACTED_TEXT VARCHAR(16777216),
    KEY_INSIGHTS VARCHAR(10000), EMBEDDING VECTOR(FLOAT, 768)
);

CREATE OR REPLACE TABLE PROCESSED.AUDIO_INTELLIGENCE (
    AUDIO_ID NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_ID NUMBER NOT NULL, FILE_NAME VARCHAR NOT NULL,
    PROCESSED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    TRANSCRIPT_TEXT VARCHAR(16777216), TRANSCRIPT_SEGMENTS VARIANT,
    AUDIO_DURATION_SECS FLOAT, SPEAKER_COUNT NUMBER,
    EXTRACTED_FIELDS VARIANT, CALL_CATEGORY VARCHAR(100),
    CALL_CATEGORY_CONFIDENCE FLOAT, SENTIMENT_SCORE FLOAT,
    SENTIMENT_DIMENSIONS VARIANT, SUMMARY VARCHAR(10000),
    DETECTED_LANGUAGE VARCHAR(50), TRANSLATED_TRANSCRIPT VARCHAR(16777216),
    CONSULTATION_NOTES VARCHAR(10000), EMBEDDING VECTOR(FLOAT, 768)
);

---
## Step 7: Create & Run AI Processing Procedures

The processing procedures apply 9-10 Cortex AI functions to each file:

| Function | What it does |
|----------|-------------|
| `AI_PARSE_DOCUMENT` | Extracts text from PDFs (OCR + layout) |
| `AI_TRANSCRIBE` | Converts audio to text |
| `AI_EXTRACT` | Pulls structured fields (patient name, diagnosis, medications) |
| `AI_CLASSIFY` | Categorizes document type |
| `SNOWFLAKE.CORTEX.SENTIMENT` | Sentiment score (-1 to 1) |
| `AI_SENTIMENT` | Multi-dimensional sentiment |
| `SNOWFLAKE.CORTEX.SUMMARIZE` | Concise summary |
| `AI_TRANSLATE` | Detects language, translates if non-English |
| `AI_REDACT` | Removes PII |
| `AI_COMPLETE` | LLM-generated insights and action items |
| `AI_EMBED` | Vector embedding for semantic search |

**First**, run the stored procedure scripts in a separate worksheet:
- `assets/05_proc_pdf.sql`
- `assets/06_proc_txt.sql`
- `assets/07_proc_audio.sql`

> **Regional note:** If `claude-3-5-sonnet` is unavailable, replace with `mistral-large2` in those scripts.

**Then** call the procedures below (takes 5-10 minutes total):

In [ ]:
CALL PROCESSED.PROCESS_PDF_FILES();

In [ ]:
CALL PROCESSED.PROCESS_TXT_FILES();

In [ ]:
CALL PROCESSED.PROCESS_AUDIO_FILES();

### Verify AI Processing

**Expected:** 6 PDFs, 4 TXT, 5 Audio — all processed.

In [ ]:
SELECT 'PDF' AS TYPE, COUNT(*) AS CNT FROM PROCESSED.PDF_INTELLIGENCE
UNION ALL SELECT 'TXT', COUNT(*) FROM PROCESSED.TXT_INTELLIGENCE
UNION ALL SELECT 'AUDIO', COUNT(*) FROM PROCESSED.AUDIO_INTELLIGENCE;

---
## Step 8: Create Cortex Search Services

**Cortex Search** provides semantic (meaning-based) retrieval over the AI-enriched content. Each search service indexes a concatenated text column and exposes filterable attributes.

We create one search service per file type so that the agent can route queries to the appropriate content.

Run `assets/11_cortex_search.sql` in a separate worksheet, then verify below.

> Wait 2-3 minutes after creation for indexing to complete.

In [ ]:
SHOW CORTEX SEARCH SERVICES IN DATABASE HEALTHCARE_AI_DEMO;

---
## Step 9: Load Structured Data

The structured data represents a typical healthcare operational database:
- **PROVIDERS** (12 rows) — Doctor directory with specialties
- **PATIENTS** (15 rows) — Patient demographics and insurance
- **CLAIMS** (30 rows) — Billing, procedures, diagnoses, status
- **APPOINTMENTS** (24 rows) — Scheduling, visit types, durations

Run `assets/09_structured_data.sql` in a separate worksheet, then verify:

In [ ]:
SELECT 'PROVIDERS' AS TBL, COUNT(*) AS CNT FROM HEALTHCARE_AI_DEMO.ANALYTICS.PROVIDERS
UNION ALL SELECT 'PATIENTS', COUNT(*) FROM HEALTHCARE_AI_DEMO.ANALYTICS.PATIENTS
UNION ALL SELECT 'CLAIMS', COUNT(*) FROM HEALTHCARE_AI_DEMO.ANALYTICS.CLAIMS
UNION ALL SELECT 'APPOINTMENTS', COUNT(*) FROM HEALTHCARE_AI_DEMO.ANALYTICS.APPOINTMENTS;

---
## Step 10: Create Semantic View

A **Semantic View** defines your data model for Cortex Analyst: tables, relationships, dimensions (categorical columns), and metrics (aggregations). This is what enables natural-language-to-SQL.

Run `assets/12_semantic_view.sql` in a separate worksheet, then verify:

In [ ]:
SHOW SEMANTIC VIEWS IN SCHEMA HEALTHCARE_AI_DEMO.ANALYTICS;

---
## Step 11: Create Cortex Agent

A **Cortex Agent** combines multiple tools into a single conversational endpoint:
- **HealthcareAnalyst** — Cortex Analyst (text-to-SQL over structured data)
- **PDFSearch** — Cortex Search over parsed PDF documents
- **TXTSearch** — Cortex Search over text documents
- **AudioSearch** — Cortex Search over transcribed audio

The agent decides which tool(s) to use based on the question.

Run `assets/13_cortex_agent.sql` in a separate worksheet, then verify:

In [ ]:
DESCRIBE AGENT HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT;

---
## Step 12: Test the Agent

### Option A: Snowflake Intelligence (recommended)

1. Go to **AI & ML → Intelligence** in the Snowsight sidebar
2. Click **New Agent**
3. Select `HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT`
4. The 4 tools are inherited automatically
5. Start chatting!

### Option B: SQL Invocation

Try these sample questions:

**Structured data query** (routes to HealthcareAnalyst → generates SQL):

In [ ]:
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
  'Which providers have the highest total billed amounts?'
);

**Unstructured search query** (routes to PDFSearch + TXTSearch):

In [ ]:
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
  'Find documents mentioning diabetes or hypertension'
);

**Audio search query** (routes to AudioSearch):

In [ ]:
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
  'What consultations discussed medication changes?'
);

**Cross-tool query** (agent uses multiple tools and combines results):

In [ ]:
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
  'What are the most common diagnoses across all medical documents?'
);

---
## Pipeline Summary

This shows the final state of all objects created during the lab.

In [ ]:
SELECT 'FILES_LOG' AS OBJECT, COUNT(*) AS ROW_COUNT FROM RAW.FILES_LOG
UNION ALL SELECT 'PDF_INTELLIGENCE', COUNT(*) FROM PROCESSED.PDF_INTELLIGENCE
UNION ALL SELECT 'TXT_INTELLIGENCE', COUNT(*) FROM PROCESSED.TXT_INTELLIGENCE
UNION ALL SELECT 'AUDIO_INTELLIGENCE', COUNT(*) FROM PROCESSED.AUDIO_INTELLIGENCE
UNION ALL SELECT 'PROVIDERS', COUNT(*) FROM ANALYTICS.PROVIDERS
UNION ALL SELECT 'PATIENTS', COUNT(*) FROM ANALYTICS.PATIENTS
UNION ALL SELECT 'CLAIMS', COUNT(*) FROM ANALYTICS.CLAIMS
UNION ALL SELECT 'APPOINTMENTS', COUNT(*) FROM ANALYTICS.APPOINTMENTS;

---
## Cleanup

Uncomment and run the lines below to remove all lab objects from your account.

In [ ]:
-- Uncomment to clean up:
-- ALTER TASK HEALTHCARE_AI_DEMO.RAW.PROCESS_NEW_FILES_TASK SUSPEND;
-- DROP DATABASE IF EXISTS HEALTHCARE_AI_DEMO;
-- DROP WAREHOUSE IF EXISTS HEALTHCARE_AI_WH;
-- DROP STORAGE INTEGRATION IF EXISTS HEALTHCARE_S3_INTEGRATION;